# pytorch-grad-camによる各種可視化手法（CIFAR100を用いたCNNの可視化）

---
## 目的
`cam.ipynb`のCAMは，「畳み込み層 → GAP → 全結合層（1層）」という特定の構造を持つネットワークにしか適用できませんでした．また，`abn.ipynb`のABNは，Attention機構をあらかじめネットワーク構造に組み込んで学習する必要がありました．

本ノートブックでは，これらの制約なしに，学習済みの任意のCNNに対して判断根拠を可視化できる`pytorch-grad-cam`ライブラリを使用し，Grad-CAM・ScoreCAM・EigenCAMなど複数の可視化手法を実行・比較する．

## pytorch-grad-camで使用する手法
- **Grad-CAM**：CAMを一般化した手法．全結合層1層という制約をなくすため，全結合層の重みの代わりに，各特徴マップに対する出力スコアの勾配をGlobal Average Poolingしたものをチャネルごとの重みとして使用する．
- **Grad-CAM++**：Grad-CAMを改良し，勾配の2次・3次の項も利用することで，同じクラスの物体が画像内に複数存在する場合などに，より高精度な可視化を実現する．
- **ScoreCAM**：勾配を一切使用しない手法．特徴マップそのものをマスクとして入力画像に適用し，そのマスクされた画像に対するネットワークの出力スコア（confidence）を重みとして使用する．勾配消失・勾配の飽和といった問題の影響を受けない．
- **EigenCAM**：特徴マップに対して主成分分析（PCA）を行い，第1主成分を可視化に用いる．クラスラベルを一切使用せず，逆伝播も不要なため，最も計算コストが低い．
- **AblationCAM**：各チャネルの特徴マップを1つずつ取り除き（ablation），出力スコアがどれだけ低下するかを重みとして使用する．勾配を使わない点はScoreCAMと似ているが，マスクではなくチャネルの除去によって重要度を測る．

いずれの手法も，最終的には入力画像と同じサイズの1チャンネルのヒートマップ（Attention Map相当）を出力する点は共通しています．

## モジュールのインポート
`pytorch-grad-cam`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q grad-cam

from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, EigenCAM, AblationCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．データセットの詳細については`classification/resnet.ipynb`を参照してください．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)
class_names = train_data.classes

print(len(train_data), len(test_data))

## ネットワークの定義と学習
`pytorch-grad-cam`はネットワーク構造に制約がないため，ここでは`cam.ipynb`と全く同じCIFAR版ResNet（BasicBlock, depth=20）を使用します（構造の詳細は`classification/resnet.ipynb`・`cam.ipynb`を参照してください）．ただし，本ノートブックのネットワークは，通常の分類モデルと同様に出力（クラススコア）のみを返すようにします（`pytorch-grad-cam`は，特徴マップの取得や勾配の計算を，指定した層に対して内部で自動的に行うフック機構を持っているため，モデル側で特徴マップを明示的に返す必要はありません）．

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.convs(x)
        out += residual
        return self.relu(out)


class ResNetBasicBlock(nn.Module):
    def __init__(self, depth, n_class=100):
        super().__init__()
        assert (depth - 2) % 6 == 0, 'depth should be 6n+2 (e.g. 20, 32, 44).'
        n_blocks = (depth - 2) // 6

        self.inplanes = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)  # 可視化のターゲットとする最後の畳み込みブロック

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * BasicBlock.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * BasicBlock.expansion),
            )
        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * BasicBlock.expansion
        for _ in range(n_blocks - 1):
            layers.append(BasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


model = ResNetBasicBlock(depth=20, n_class=100).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

batch_size = 128
epoch_num = 20
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)
criterion = nn.CrossEntropyLoss().to(device)

model.train()
train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0
    for image, label in train_loader:
        image, label = image.to(device), label.to(device)

        y = model(image)
        loss = criterion(y, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print(f'epoch: {epoch}, mean loss: {sum_loss / len(train_loader):.4f}, mean accuracy: {count.item() / len(train_data):.4f}, elapsed_time: {time() - train_start:.4f}')

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()
count = 0
with torch.no_grad():
    for image, label in test_loader:
        image, label = image.to(device), label.to(device)
        y = model(image)
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## 複数の可視化手法の比較
`pytorch-grad-cam`の各手法は共通のインターフェースを持っており，`手法名(model=model, target_layers=target_layers)`でオブジェクトを作成した後，`cam(input_tensor=..., targets=...)`を呼び出すだけで，どの手法でも同じようにヒートマップ（0〜1に正規化済み，入力画像と同じ解像度）を取得できます．`target_layers`には，可視化対象とする畳み込み層（ここでは最後のResidual Blockである`model.layer3[-1]`）を指定します．`targets`には，どのクラスに対する可視化を行うかを`ClassifierOutputTarget`で指定します（`EigenCAM`はクラスを使用しないため`targets=None`でも動作します）．

In [ ]:
target_layers = [model.layer3[-1]]

cam_methods = {
    'GradCAM': GradCAM(model=model, target_layers=target_layers),
    'GradCAM++': GradCAMPlusPlus(model=model, target_layers=target_layers),
    'ScoreCAM': ScoreCAM(model=model, target_layers=target_layers),
    'EigenCAM': EigenCAM(model=model, target_layers=target_layers),
    'AblationCAM': AblationCAM(model=model, target_layers=target_layers),
}

image, label = test_data[0]
input_tensor = image.unsqueeze(0).to(device)
pred_class = torch.argmax(model(input_tensor), dim=1).item()
targets = [ClassifierOutputTarget(pred_class)]

image_np = image.permute(1, 2, 0).numpy().astype(np.float32)

fig, axes = plt.subplots(1, len(cam_methods) + 1, figsize=(2.4 * (len(cam_methods) + 1), 2.6))
axes[0].imshow(image_np); axes[0].axis('off')
axes[0].set_title(f'input\nGT: {class_names[label]}', fontsize=9)

for ax, (name, cam) in zip(axes[1:], cam_methods.items()):
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]  # (H, W), 0〜1に正規化済み
    visualization = show_cam_on_image(image_np, grayscale_cam, use_rgb=True)
    ax.imshow(visualization); ax.axis('off')
    ax.set_title(name, fontsize=9)

plt.tight_layout()
plt.show()

## GradCAMによる複数画像の可視化
続いて，1つの手法（GradCAM）に絞り，複数の画像に対する可視化結果をまとめて確認します．

In [ ]:
gradcam = GradCAM(model=model, target_layers=target_layers)

model.eval()
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(2.2 * n_show, 4.6))

for i in range(n_show):
    image, label = test_data[i]
    input_tensor = image.unsqueeze(0).to(device)

    with torch.no_grad():
        pred_class = torch.argmax(model(input_tensor), dim=1).item()

    # GradCAMは内部でbackward()を呼び出すため，torch.no_grad()の外で実行する
    grayscale_cam = gradcam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])[0]

    image_np = image.permute(1, 2, 0).numpy().astype(np.float32)
    visualization = show_cam_on_image(image_np, grayscale_cam, use_rgb=True)

    axes[0, i].imshow(image_np); axes[0, i].axis('off')
    axes[0, i].set_title(f'GT: {class_names[label]}', fontsize=9)
    axes[1, i].imshow(visualization); axes[1, i].axis('off')
    axes[1, i].set_title(f'pred: {class_names[pred_class]}', fontsize=9)

plt.tight_layout()
plt.show()

## 課題

1. `target_layers`を`model.layer3[-1]`から`model.layer2[-1]`や`model.layer1[-1]`に変更し，可視化されるヒートマップの解像度や見た目がどのように変化するか確認してください．
2. `cam.ipynb`で実装したCAMと，本ノートブックのGradCAMを同じ画像・同じネットワーク構造で比較し，両者がどれくらい似た結果になるか確認してください（GradCAMは，GAP+全結合層1層という構造のもとではCAMと理論的に等価になることが知られています）．
3. ScoreCAMやAblationCAMは，GradCAMと比べて計算に時間がかかります．その理由を，それぞれの手法の仕組みから説明してください．